# 04 — Thesis Figures (Publication Quality)

**Thesis:** Macroeconomic Factor-Based Dynamic Portfolio Optimization

All figures are saved to `data/results/` in PDF format at 300 DPI.

| Figure | Description |
|--------|-------------|
| Fig 1 | Asset price evolution & normalized series |
| Fig 2 | Asset correlation heatmap |
| Fig 3 | Macroeconomic indicators time series |
| Fig 4 | Macro-asset lead-lag correlation |
| Fig 5 | Model comparison (RMSE, R²) |
| Fig 6 | Feature importance (top features) |
| Fig 7 | Efficient frontier with named portfolios |
| Fig 8 | Sensitivity analysis (λ vs weights) |
| Fig 9 | Cumulative returns comparison |
| Fig 10 | Portfolio weights over time |
| Fig 11 | Drawdown analysis |
| Fig 12 | Rolling Sharpe ratio |
| Fig 13 | Stress test results |
| Fig 14 | Performance metrics table |

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from src.config import (
    RAW_DIR, PROCESSED_DIR, RESULTS_DIR,
    TICKERS, TICKER_LIST, FRED_SERIES,
    STRESS_SCENARIOS, ASSET_CLASSES,
    FIGURE_DPI, FIGURE_FORMAT, PREDICTION_HORIZON,
)
from src.visualization.plots import save_figure

# Thesis-quality settings
sns.set_theme(style='whitegrid', palette='deep', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': FIGURE_DPI,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.fontsize': 9,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'figure.figsize': (12, 6),
})

# Load data
prices = pd.read_csv(RAW_DIR / 'prices.csv', index_col=0, parse_dates=True)
macro = pd.read_csv(RAW_DIR / 'macro.csv', index_col=0, parse_dates=True)
returns = prices.pct_change().dropna()

print(f'Figures will be saved to: {RESULTS_DIR}')
print(f'Format: {FIGURE_FORMAT}, DPI: {FIGURE_DPI}')

## Figure 1: Asset Price Evolution

In [ ]:
normalized = prices / prices.iloc[0] * 100
fig, ax = plt.subplots(figsize=(12, 7))

for col in normalized.columns:
    ax.plot(normalized.index, normalized[col], label=f'{col} ({TICKERS[col]})', linewidth=1.2)

# Shade crises
for name, (start, end) in STRESS_SCENARIOS.items():
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.08, color='red')

ax.set_title('Normalized Asset Prices (Base = 100)')
ax.set_ylabel('Indexed Value (Log Scale)')
ax.set_xlabel('Date')
ax.set_yscale('log')
ax.legend(loc='upper left', fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
save_figure(fig, 'fig01_price_evolution')

## Figure 2: Asset Correlation Heatmap

In [ ]:
corr = returns.corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8, 'label': 'Correlation'})
ax.set_title('Asset Return Correlation Matrix')
save_figure(fig, 'fig02_correlation_heatmap')

## Figure 3: Macroeconomic Indicators

In [ ]:
n_series = len(macro.columns)
fig, axes = plt.subplots(n_series, 1, figsize=(12, 2.5 * n_series), sharex=True)

for ax, col in zip(axes, macro.columns):
    data = macro[col].dropna()
    ax.plot(data.index, data.values, linewidth=0.8, color='steelblue')
    ax.set_ylabel(FRED_SERIES.get(col, col), fontsize=8)
    ax.tick_params(labelsize=7)
    for name, (start, end) in STRESS_SCENARIOS.items():
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.08, color='red')

axes[0].set_title('FRED Macroeconomic Indicators')
axes[-1].set_xlabel('Date')
fig.tight_layout()
save_figure(fig, 'fig03_macro_indicators')

## Figure 4: Macro-Asset Lead-Lag Correlation

In [ ]:
monthly_returns = returns.resample('ME').sum()
monthly_macro = macro.resample('ME').last().pct_change()
common_idx = monthly_returns.index.intersection(monthly_macro.index)

macro_lead = monthly_macro.loc[common_idx].shift(1).dropna()
mr_aligned = monthly_returns.loc[macro_lead.index]

cross_corr = pd.DataFrame(
    np.corrcoef(macro_lead.fillna(0).T, mr_aligned.fillna(0).T)[:len(macro_lead.columns), len(macro_lead.columns):],
    index=[FRED_SERIES.get(c, c) for c in macro_lead.columns],
    columns=mr_aligned.columns
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(cross_corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            linewidths=0.5, vmin=-0.3, vmax=0.3, ax=ax,
            cbar_kws={'shrink': 0.8, 'label': 'Correlation'})
ax.set_title('Macro Change (t−1) vs Asset Return (t) — Monthly Lead-Lag Correlation')
save_figure(fig, 'fig04_macro_asset_correlation')

## Figure 5: Model Comparison

In [ ]:
try:
    model_comp = pd.read_csv(RESULTS_DIR / 'model_comparison.csv')
    
    # Average across assets
    avg_comp = model_comp.groupby('model')[['rmse_mean', 'r2_mean']].mean().sort_values('rmse_mean')
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # RMSE
    avg_comp['rmse_mean'].plot.barh(ax=axes[0], color='steelblue')
    axes[0].set_title('Average RMSE Across Assets (lower is better)')
    axes[0].set_xlabel('RMSE')
    
    # R²
    colors = ['green' if v > 0 else 'red' for v in avg_comp['r2_mean']]
    avg_comp['r2_mean'].plot.barh(ax=axes[1], color=colors)
    axes[1].axvline(x=0, color='black', linewidth=0.5)
    axes[1].set_title('Average R² Across Assets (higher is better)')
    axes[1].set_xlabel('R²')
    
    fig.suptitle('Predictive Model Comparison', fontsize=14)
    plt.tight_layout()
    save_figure(fig, 'fig05_model_comparison')
except FileNotFoundError:
    print('Model comparison not found. Run training step first.')

## Figure 6: Feature Importance

In [ ]:
try:
    import pickle
    from src.models.trainer import get_feature_importance
    
    # Load best model for SPY
    model_files = list(RESULTS_DIR.glob('model_xgboost_SPY*.pkl'))
    if model_files:
        with open(model_files[0], 'rb') as f:
            result = pickle.load(f)
        
        features_df = pd.read_csv(PROCESSED_DIR / 'features.csv', index_col=0, parse_dates=True)
        feature_cols = [c for c in features_df.columns if not c.endswith(f'_ret_{PREDICTION_HORIZON}d')]
        
        fi = get_feature_importance(result['model'], feature_cols, 'xgboost')
        top = fi.head(20).iloc[::-1]
        
        fig, ax = plt.subplots(figsize=(10, 8))
        ax.barh(top['feature'], top['importance_pct'], color='steelblue')
        ax.set_title('Top 20 Features — XGBoost (SPY 21-Day Return Prediction)')
        ax.set_xlabel('Importance (%)')
        save_figure(fig, 'fig06_feature_importance')
    else:
        print('No trained model found for SPY.')
except Exception as e:
    print(f'Feature importance figure skipped: {e}')

## Figure 7: Efficient Frontier

In [ ]:
from src.optimization.optimizer import (
    mean_variance_optimize, max_sharpe_optimize,
    minimum_variance_optimize, risk_parity_optimize,
    efficient_frontier, estimate_covariance,
)

mu = returns.mean().values * 252
cov = estimate_covariance(returns, method='shrinkage')
ef = efficient_frontier(mu, cov, n_points=100)

# Named portfolios
named = {}
w = max_sharpe_optimize(mu, cov)
if w is not None:
    named['Max Sharpe'] = (np.sqrt(w @ cov @ w), mu @ w, '*', 200)
w = minimum_variance_optimize(cov)
if w is not None:
    named['Min Variance'] = (np.sqrt(w @ cov @ w), mu @ w, 'D', 150)
w = risk_parity_optimize(cov)
named['Risk Parity'] = (np.sqrt(w @ cov @ w), mu @ w, 's', 150)
w_eq = np.ones(len(TICKER_LIST)) / len(TICKER_LIST)
named['Equal Weight'] = (np.sqrt(w_eq @ cov @ w_eq), mu @ w_eq, 'o', 120)

fig, ax = plt.subplots(figsize=(12, 8))
ax.plot(ef['volatility'] * 100, ef['return'] * 100, 'b-', linewidth=2.5, label='Efficient Frontier', zorder=1)

# Individual assets
for i, ticker in enumerate(TICKER_LIST):
    vol = np.sqrt(cov[i, i]) * 100
    ret = mu[i] * 100
    ax.scatter(vol, ret, s=60, zorder=3, color='gray', alpha=0.7)
    ax.annotate(ticker, (vol, ret), textcoords='offset points', xytext=(5, 5), fontsize=8, color='gray')

# Named portfolios
for name, (vol, ret, marker, size) in named.items():
    ax.scatter(vol * 100, ret * 100, s=size, marker=marker, zorder=5, label=name, edgecolors='black', linewidths=1)

ax.set_title('Mean-Variance Efficient Frontier')
ax.set_xlabel('Annualized Volatility (%)')
ax.set_ylabel('Annualized Expected Return (%)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
save_figure(fig, 'fig07_efficient_frontier')

## Figure 8: Sensitivity Analysis (λ vs Weights)

In [ ]:
lambdas = np.linspace(0.1, 20, 60)
weight_paths = {t: [] for t in TICKER_LIST}

for lam in lambdas:
    w = mean_variance_optimize(mu, cov, risk_aversion=lam)
    for i, t in enumerate(TICKER_LIST):
        weight_paths[t].append(w[i] if w is not None else np.nan)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Individual assets
for t in TICKER_LIST:
    axes[0].plot(lambdas, weight_paths[t], label=t, linewidth=1.2)
axes[0].set_title('Individual Asset Weights vs λ')
axes[0].set_xlabel('Risk Aversion (λ)')
axes[0].set_ylabel('Portfolio Weight')
axes[0].legend(fontsize=7, ncol=2)

# Asset classes
for cls, tickers in ASSET_CLASSES.items():
    cls_w = [sum(weight_paths[t][i] for t in tickers) for i in range(len(lambdas))]
    axes[1].plot(lambdas, cls_w, linewidth=2.5, label=cls)
axes[1].set_title('Asset Class Allocation vs λ')
axes[1].set_xlabel('Risk Aversion (λ)')
axes[1].set_ylabel('Total Weight')
axes[1].legend()

fig.suptitle('Sensitivity of Optimal Allocation to Risk Aversion Parameter', fontsize=13)
plt.tight_layout()
save_figure(fig, 'fig08_sensitivity_lambda')

## Figures 9–14: Backtest Results

In [ ]:
from src.optimization.backtester import (
    backtest_strategy, benchmark_equal_weight,
    compare_strategies, stress_test, rolling_sharpe,
    compute_portfolio_metrics,
)

# Run backtests
strategies = {
    'MV (λ=2)': {'risk_aversion': 2.0},
    'MV (λ=5)': {'risk_aversion': 5.0},
    'MV (λ=10)': {'risk_aversion': 10.0},
}

print('Running backtests for thesis figures...')
comparison = compare_strategies(prices, strategies=strategies)
results = comparison['results']
metrics_df = comparison['metrics']

best = metrics_df['sharpe_ratio'].idxmax()
print(f'Best strategy by Sharpe: {best}')

In [ ]:
# Figure 9: Cumulative Returns
fig, ax = plt.subplots(figsize=(12, 7))

for name, result in results.items():
    cum = (1 + result['returns']).cumprod()
    ax.plot(cum.index, cum.values, label=name, linewidth=1.5)

for name, (start, end) in STRESS_SCENARIOS.items():
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.08, color='red')

ax.set_title('Cumulative Returns — Strategy Comparison')
ax.set_ylabel('Growth of $1')
ax.set_xlabel('Date')
ax.legend(loc='upper left')
ax.set_yscale('log')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.1f'))
ax.grid(True, alpha=0.3)
save_figure(fig, 'fig09_cumulative_returns')

In [ ]:
# Figure 10: Portfolio Weights Over Time
best_weights = results[best]['weights']

fig, ax = plt.subplots(figsize=(12, 6))
ax.stackplot(best_weights.index, best_weights.T.values, labels=best_weights.columns, alpha=0.85)
ax.set_title(f'Portfolio Weights Over Time — {best}')
ax.set_ylabel('Weight')
ax.set_xlabel('Date')
ax.set_ylim(0, 1)
ax.legend(loc='upper left', fontsize=8, ncol=2)
save_figure(fig, 'fig10_weights_over_time')

In [ ]:
# Figure 11: Drawdown Analysis
best_ret = results[best]['returns']
cum = (1 + best_ret).cumprod()
dd = (cum / cum.cummax()) - 1

fig, axes = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={'height_ratios': [2, 1]})

axes[0].plot(cum.index, cum.values, linewidth=1.5, color='steelblue')
axes[0].set_title(f'Cumulative Return & Drawdowns — {best}')
axes[0].set_ylabel('Growth of $1')

axes[1].fill_between(dd.index, dd.values, 0, alpha=0.6, color='red')
axes[1].set_ylabel('Drawdown')
axes[1].set_xlabel('Date')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

for ax in axes:
    for name, (start, end) in STRESS_SCENARIOS.items():
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.08, color='gray')

plt.tight_layout()
save_figure(fig, 'fig11_drawdowns')

In [ ]:
# Figure 12: Rolling Sharpe Ratio
fig, ax = plt.subplots(figsize=(12, 6))

for name, result in results.items():
    rs = rolling_sharpe(result['returns'], window=252)
    if len(rs) > 0:
        ax.plot(rs.index, rs.values, label=name, linewidth=1.2)

ax.axhline(y=0, color='black', linewidth=0.5, linestyle='--')
ax.axhline(y=1, color='green', linewidth=0.5, linestyle=':', alpha=0.5, label='Sharpe = 1')
ax.set_title('Rolling 252-Day Sharpe Ratio')
ax.set_ylabel('Sharpe Ratio')
ax.set_xlabel('Date')
ax.legend(loc='lower left', fontsize=8)

for name, (start, end) in STRESS_SCENARIOS.items():
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.08, color='red')

save_figure(fig, 'fig12_rolling_sharpe')

In [ ]:
# Figure 13: Stress Test
eq_ret = results['Equal Weight']['returns']
stress_df = stress_test(results[best]['returns'], benchmark_returns=eq_ret)

if len(stress_df) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(stress_df))
    width = 0.35
    
    ax.bar(x - width/2, stress_df['total_return'] * 100, width, label=best, color='steelblue')
    if 'benchmark_return' in stress_df.columns:
        ax.bar(x + width/2, stress_df['benchmark_return'] * 100, width, label='Equal Weight', color='coral')
    
    ax.set_xticks(x)
    labels = [f"{idx}\n({stress_df.loc[idx, 'start']} to {stress_df.loc[idx, 'end']})" 
              for idx in stress_df.index]
    ax.set_xticklabels(labels, fontsize=8, rotation=30, ha='right')
    ax.set_title('Portfolio Performance During Historical Stress Periods')
    ax.set_ylabel('Total Return (%)')
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.legend()
    save_figure(fig, 'fig13_stress_test')

In [ ]:
# Figure 14: Performance Metrics Table
fig, ax = plt.subplots(figsize=(14, max(3, len(metrics_df) * 0.6 + 2)))
ax.axis('off')

display_df = metrics_df.copy()
col_formats = {
    'annualized_return': lambda x: f'{x:.2%}',
    'annualized_volatility': lambda x: f'{x:.2%}',
    'sharpe_ratio': lambda x: f'{x:.3f}',
    'max_drawdown': lambda x: f'{x:.2%}',
    'calmar_ratio': lambda x: f'{x:.3f}',
    'total_return': lambda x: f'{x:.2%}',
}
for col, fmt in col_formats.items():
    if col in display_df.columns:
        display_df[col] = display_df[col].apply(fmt)

# Rename columns for thesis
rename = {
    'annualized_return': 'Ann. Return',
    'annualized_volatility': 'Ann. Vol',
    'sharpe_ratio': 'Sharpe',
    'max_drawdown': 'Max DD',
    'calmar_ratio': 'Calmar',
    'total_return': 'Total Return',
}
display_df = display_df.rename(columns=rename)

table = ax.table(
    cellText=display_df.values,
    colLabels=display_df.columns,
    rowLabels=display_df.index,
    cellLoc='center',
    loc='center',
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.3, 1.6)

# Style header
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(fontweight='bold')
        cell.set_facecolor('#4472C4')
        cell.set_text_props(color='white', fontweight='bold')
    if col == -1:
        cell.set_text_props(fontweight='bold')

ax.set_title('Backtest Performance Summary', fontsize=14, pad=20)
save_figure(fig, 'fig14_metrics_table')

In [ ]:
print('All thesis figures generated!')
print(f'Location: {RESULTS_DIR}')

import os
fig_files = sorted([f for f in os.listdir(RESULTS_DIR) if f.startswith('fig')])
for f in fig_files:
    print(f'  {f}')